In [7]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import time
from datetime import datetime
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn imports
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split, cross_validate
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score, roc_auc_score, 
    accuracy_score, precision_score, recall_score, roc_curve, precision_recall_curve
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import VotingClassifier

# Algoritmos para tuning RECALL-focused
from sklearn.svm import SVC
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from sklearn.ensemble import RandomForestClassifier

print("🚀 TUNING 2.0 - RECALL FOCUSED + ENSEMBLE PARANÓICO")
print("🎯 OBJETIVO: 90%+ RECALL - NÃO PERDER NENHUM BEBÊ!")
print("=" * 70)

# Carregar dados
train_path = "/Users/marcelosilva/Desktop/projectOne/6/B- Binary Target DS DT/DatasetTrain.csv"
df_train = pd.read_csv(train_path)

print(f"✅ Dados carregados: {df_train.shape}")

# Target
target_col = 'status_nutricional_who'
target_counts = df_train[target_col].value_counts()
print(f"✅ Target distribution: {target_counts.to_dict()}")
print(f"✅ Classe minoritária: {target_counts[1]/len(df_train)*100:.1f}% ({target_counts[1]} bebês)")

# Limpar dados
exclude_cols = ['id_anon', 'vd_zimc']
df_clean = df_train.drop(columns=exclude_cols, errors='ignore')

# TOP 3 FEATURES (comprovadamente as melhores)
top_3_features = ['imc_pre_gravidico', 'k06_peso_engravidar', 'h02_peso']

print(f"✅ Features selecionadas: {top_3_features}")
print(f"📊 Estratégia: Máxima sensibilidade com apenas 3 features")

# Função de preprocessamento otimizada
def preprocess_data(df, features, target):
    """Preprocessa os dados com foco em recall"""
    
    # Verificar features disponíveis
    available_features = [f for f in features if f in df.columns]
    if len(available_features) != len(features):
        missing = set(features) - set(available_features)
        print(f"⚠️  Features não encontradas: {missing}")
        features = available_features
    
    # Dataset com features + target
    df_subset = df[features + [target]].copy()
    
    # Tratar valores faltantes (estratégia conservadora)
    numeric_features = df_subset.select_dtypes(include=[np.number]).columns.tolist()
    if target in numeric_features:
        numeric_features.remove(target)
    
    categorical_features = df_subset.select_dtypes(include=['object']).columns.tolist()
    
    # Imputação numérica (mediana é mais robusta)
    if len(numeric_features) > 0:
        imputer_num = SimpleImputer(strategy='median')
        df_subset[numeric_features] = imputer_num.fit_transform(df_subset[numeric_features])
    
    # Encoding categórico
    for col in categorical_features:
        le = LabelEncoder()
        df_subset[col] = df_subset[col].fillna('Missing')
        df_subset[col] = le.fit_transform(df_subset[col])
    
    X = df_subset.drop(columns=[target])
    y = df_subset[target]
    
    return X, y

# Preprocessar dados
X, y = preprocess_data(df_clean, top_3_features, target_col)

# Escalar features (crítico para SVM)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"✅ Dataset preprocessado: {X.shape}")
print(f"✅ Features escaladas para SVM")

# Cross-validation strategy
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ALGORITMOS PARA TUNING RECALL-FOCUSED
algorithms_recall = {
    'SVC_linear': {
        'model': SVC(kernel='linear', probability=True, random_state=42),
        'params': {
            'C': [0.01, 0.1, 1, 10, 100],
            'class_weight': [
                {0: 1, 1: 5},   # Moderado
                {0: 1, 1: 8},   # Alto
                {0: 1, 1: 12},  # Muito alto
                {0: 1, 1: 15}   # EXTREMO
            ]
        }
    },
    
    'Ridge_Recall': {
        'model': RidgeClassifier(random_state=42),
        'params': {
            'alpha': [0.001, 0.01, 0.1, 1, 10],  # Menor regularização
            'class_weight': [
                {0: 1, 1: 5},
                {0: 1, 1: 8}, 
                {0: 1, 1: 12},
                {0: 1, 1: 15}
            ]
        }
    },
    
    'LogisticRegression_Recall': {
        'model': LogisticRegression(random_state=42, max_iter=2000),
        'params': {
            'C': [0.01, 0.1, 1, 10, 100],
            'penalty': ['l1', 'l2'],
            'solver': ['liblinear'],  # Compatível com l1 e l2
            'class_weight': [
                {0: 1, 1: 5},
                {0: 1, 1: 8},
                {0: 1, 1: 12},
                {0: 1, 1: 15}
            ]
        }
    },
    
    'RandomForest_Recall': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [100, 200],
            'max_depth': [3, 5, 7],        # Shallow trees = mais sensível
            'min_samples_leaf': [1, 2, 3], # Menos samples = mais sensível
            'min_samples_split': [2, 3, 5],
            'class_weight': [
                {0: 1, 1: 8},
                {0: 1, 1: 12},
                {0: 1, 1: 15}
            ]
        }
    }
}

# Mostrar configuração dos algoritmos
print(f"\n🧪 ALGORITMOS PARA TUNING RECALL:")
for alg_name, alg_config in algorithms_recall.items():
    param_count = 1
    for param_name, param_values in alg_config['params'].items():
        param_count *= len(param_values)
    print(f"  - {alg_name}: {param_count} combinações")

print(f"\n🔥 INICIANDO TUNING RECALL-FOCUSED...")
print(f"⏱️ Tempo estimado: 60-120 minutos")

# Executar Grid Search RECALL-focused
tuning_results = []
algorithm_count = 0

print(f"\n🧪 COMEÇANDO TUNING...")
print("=" * 70)

start_time = time.time()

for alg_name, alg_config in algorithms_recall.items():
    algorithm_count += 1
    
    print(f"\n🔧 {algorithm_count}/4 | Tuning {alg_name} para MÁXIMO RECALL")
    
    try:
        start_alg_time = time.time()
        
        # Grid Search com foco em RECALL
        model = alg_config['model']
        param_grid = alg_config['params']
        
        # Grid Search otimizado para recall
        grid_search = GridSearchCV(
            model, param_grid, 
            cv=cv_strategy, 
            scoring='recall',  # ← FOCO EM RECALL!
            n_jobs=-1, 
            verbose=0
        )
        
        grid_search.fit(X_scaled, y)
        
        end_alg_time = time.time()
        tuning_time = end_alg_time - start_alg_time
        
        # Avaliar melhor modelo com múltiplas métricas
        best_model = grid_search.best_estimator_
        
        cv_results = cross_validate(
            best_model, X_scaled, y, cv=cv_strategy,
            scoring=['accuracy', 'f1', 'precision', 'recall', 'roc_auc'],
            return_train_score=False,
            n_jobs=-1
        )
        
        # Salvar resultados
        result = {
            'algorithm': alg_name,
            'best_recall_score': grid_search.best_score_,
            'best_params': str(grid_search.best_params_),
            'tuning_time': tuning_time,
            
            # Métricas detalhadas
            'cv_accuracy_mean': cv_results['test_accuracy'].mean(),
            'cv_accuracy_std': cv_results['test_accuracy'].std(),
            'cv_f1_mean': cv_results['test_f1'].mean(),
            'cv_f1_std': cv_results['test_f1'].std(),
            'cv_precision_mean': cv_results['test_precision'].mean(),
            'cv_precision_std': cv_results['test_precision'].std(),
            'cv_recall_mean': cv_results['test_recall'].mean(),
            'cv_recall_std': cv_results['test_recall'].std(),
            'cv_auc_mean': cv_results['test_roc_auc'].mean(),
            'cv_auc_std': cv_results['test_roc_auc'].std()
        }
        
        tuning_results.append(result)
        
        print(f"   ✅ RECALL: {grid_search.best_score_:.4f} | F1: {cv_results['test_f1'].mean():.4f} | AUC: {cv_results['test_roc_auc'].mean():.4f}")
        print(f"   ⏱️  Tempo: {tuning_time:.1f}s")
        print(f"   🎯 Params: {grid_search.best_params_}")
        
    except Exception as e:
        print(f"   ❌ ERRO: {str(e)[:100]}")
        
        tuning_results.append({
            'algorithm': alg_name,
            'best_recall_score': 0,
            'error': str(e)[:200],
            'tuning_time': 0
        })

# Análise dos resultados
tuning_df = pd.DataFrame(tuning_results)
success_df = tuning_df[tuning_df['best_recall_score'] > 0].copy()

total_time = time.time() - start_time

print(f"\n⏱️  TUNING CONCLUÍDO em {total_time/60:.1f} minutos")
print("=" * 70)

if len(success_df) > 0:
    # Ordenar por RECALL
    success_df = success_df.sort_values('cv_recall_mean', ascending=False)
    
    print(f"\n📈 RESULTADOS TUNING RECALL-FOCUSED:")
    print(f"Algoritmos bem-sucedidos: {len(success_df)}/4")
    
    print(f"\n🏆 RANKING POR RECALL:")
    for i, (_, row) in enumerate(success_df.iterrows(), 1):
        print(f"{i}. {row['algorithm']:25s} | RECALL: {row['cv_recall_mean']:.4f} (±{row['cv_recall_std']:.3f})")
        print(f"   F1: {row['cv_f1_mean']:.4f} | AUC: {row['cv_auc_mean']:.4f} | Acc: {row['cv_accuracy_mean']:.4f}")
        print(f"   Params: {row['best_params']}")
        print()

# THRESHOLD OPTIMIZATION - Buscar threshold ótimo para máximo recall
print(f"\n🎯 THRESHOLD OPTIMIZATION - BUSCA POR 90%+ RECALL")
print("=" * 70)

threshold_results = []

# Testar thresholds nos TOP 3 modelos
top_3_models = success_df.head(3) if len(success_df) >= 3 else success_df

for _, model_info in top_3_models.iterrows():
    alg_name = model_info['algorithm']
    print(f"\n🔍 Otimizando threshold para {alg_name}")
    
    try:
        # Recriar melhor modelo
        alg_config = algorithms_recall[alg_name]
        model = alg_config['model']
        best_params = eval(model_info['best_params'])
        
        # Configurar modelo com melhores parâmetros
        model.set_params(**best_params)
        
        # Split para threshold optimization
        X_train, X_val, y_train, y_val = train_test_split(
            X_scaled, y, test_size=0.3, random_state=42, stratify=y
        )
        
        # Treinar modelo
        model.fit(X_train, y_train)
        
        # Get probabilities
        if hasattr(model, 'predict_proba'):
            y_proba = model.predict_proba(X_val)[:, 1]
        elif hasattr(model, 'decision_function'):
            # Para SVM e outros
            from sklearn.calibration import CalibratedClassifierCV
            calibrated_model = CalibratedClassifierCV(model, cv=3)
            calibrated_model.fit(X_train, y_train)
            y_proba = calibrated_model.predict_proba(X_val)[:, 1]
        else:
            print(f"   ⚠️  {alg_name} não suporta probabilidades")
            continue
        
        # Testar diferentes thresholds
        thresholds = np.arange(0.1, 0.6, 0.05)
        best_threshold = 0.5
        best_recall = 0
        
        for threshold in thresholds:
            y_pred = (y_proba >= threshold).astype(int)
            recall = recall_score(y_val, y_pred)
            precision = precision_score(y_val, y_pred) if sum(y_pred) > 0 else 0
            f1 = f1_score(y_val, y_pred)
            
            if recall >= 0.85:  # Se conseguir 85%+ recall
                print(f"   ✅ Threshold {threshold:.2f}: Recall {recall:.4f} | F1 {f1:.4f} | Precision {precision:.4f}")
                
                if recall > best_recall:
                    best_recall = recall
                    best_threshold = threshold
        
        # Salvar melhor resultado
        if best_recall > 0:
            y_pred_best = (y_proba >= best_threshold).astype(int)
            threshold_results.append({
                'algorithm': alg_name,
                'best_threshold': best_threshold,
                'threshold_recall': best_recall,
                'threshold_f1': f1_score(y_val, y_pred_best),
                'threshold_precision': precision_score(y_val, y_pred_best),
                'threshold_accuracy': accuracy_score(y_val, y_pred_best)
            })
            
            print(f"   🎯 MELHOR: Threshold {best_threshold:.2f} → Recall {best_recall:.4f}")
        else:
            print(f"   ❌ Não conseguiu atingir 85%+ recall")
    
    except Exception as e:
        print(f"   ❌ Erro: {str(e)[:100]}")

# ENSEMBLE PARANÓICO
print(f"\n🤖 ENSEMBLE PARANÓICO - COMBINANDO MODELOS")
print("=" * 70)

if len(threshold_results) >= 2:
    # Preparar modelos para ensemble
    ensemble_models = []
    ensemble_names = []
    
    for result in threshold_results[:3]:  # TOP 3
        alg_name = result['algorithm']
        alg_config = algorithms_recall[alg_name]
        model = alg_config['model']
        
        # Buscar melhores parâmetros
        model_row = success_df[success_df['algorithm'] == alg_name].iloc[0]
        best_params = eval(model_row['best_params'])
        
        model.set_params(**best_params)
        ensemble_models.append((alg_name, model))
        ensemble_names.append(alg_name)
    
    print(f"🔗 Criando ensemble com: {ensemble_names}")
    
    # Teste diferentes estratégias de ensemble
    ensemble_strategies = {
        'voting_soft': VotingClassifier(estimators=ensemble_models, voting='soft'),
        'voting_hard': VotingClassifier(estimators=ensemble_models, voting='hard')
    }
    
    ensemble_results = []
    
    for strategy_name, ensemble_model in ensemble_strategies.items():
        print(f"\n🧪 Testando {strategy_name}")
        
        try:
            # Cross-validation do ensemble
            cv_results = cross_validate(
                ensemble_model, X_scaled, y, cv=cv_strategy,
                scoring=['accuracy', 'f1', 'precision', 'recall', 'roc_auc'],
                return_train_score=False,
                n_jobs=-1
            )
            
            ensemble_results.append({
                'strategy': strategy_name,
                'ensemble_recall': cv_results['test_recall'].mean(),
                'ensemble_f1': cv_results['test_f1'].mean(),
                'ensemble_precision': cv_results['test_precision'].mean(),
                'ensemble_accuracy': cv_results['test_accuracy'].mean(),
                'ensemble_auc': cv_results['test_roc_auc'].mean()
            })
            
            print(f"   ✅ RECALL: {cv_results['test_recall'].mean():.4f}")
            print(f"   📊 F1: {cv_results['test_f1'].mean():.4f} | AUC: {cv_results['test_roc_auc'].mean():.4f}")
            
        except Exception as e:
            print(f"   ❌ Erro: {str(e)[:80]}")

# RESULTADOS FINAIS
print(f"\n🏆 RESUMO FINAL - MÁXIMO RECALL ACHIEVED")
print("=" * 70)

if len(success_df) > 0:
    best_individual = success_df.iloc[0]
    print(f"🥇 MELHOR MODELO INDIVIDUAL:")
    print(f"   {best_individual['algorithm']}")
    print(f"   RECALL: {best_individual['cv_recall_mean']:.4f} (±{best_individual['cv_recall_std']:.3f})")
    print(f"   F1: {best_individual['cv_f1_mean']:.4f} | AUC: {best_individual['cv_auc_mean']:.4f}")

if len(threshold_results) > 0:
    best_threshold = max(threshold_results, key=lambda x: x['threshold_recall'])
    print(f"\n🎯 MELHOR COM THRESHOLD OTIMIZADO:")
    print(f"   {best_threshold['algorithm']} com threshold {best_threshold['best_threshold']:.2f}")
    print(f"   RECALL: {best_threshold['threshold_recall']:.4f}")
    print(f"   F1: {best_threshold['threshold_f1']:.4f}")

if len(ensemble_results) > 0:
    best_ensemble = max(ensemble_results, key=lambda x: x['ensemble_recall'])
    print(f"\n🤖 MELHOR ENSEMBLE:")
    print(f"   Estratégia: {best_ensemble['strategy']}")
    print(f"   RECALL: {best_ensemble['ensemble_recall']:.4f}")
    print(f"   F1: {best_ensemble['ensemble_f1']:.4f}")

# Interpretação médica
if len(success_df) > 0:
    best_recall = success_df.iloc[0]['cv_recall_mean']
    babies_detected = int(849 * best_recall)
    babies_missed = 849 - babies_detected
    
    print(f"\n👶 IMPACTO MÉDICO:")
    print(f"   Dos 849 bebês com problema nutricional:")
    print(f"   ✅ DETECTADOS: {babies_detected} bebês ({best_recall*100:.1f}%)")
    print(f"   ❌ PERDIDOS: {babies_missed} bebês ({(1-best_recall)*100:.1f}%)")
    
    if best_recall >= 0.80:
        print(f"   🎯 META ATINGIDA: Excelente sensibilidade para triagem médica!")
    elif best_recall >= 0.70:
        print(f"   ⚠️  Meta parcial: Boa sensibilidade, pode melhorar")
    else:
        print(f"   ❌ Meta não atingida: Sensibilidade ainda baixa")

# Salvar resultados
output_dir = "/Users/marcelosilva/Desktop/projectOne/6/E-Sklearn"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

os.makedirs(output_dir, exist_ok=True)

# Salvar todos os resultados
tuning_df.to_csv(f"{output_dir}/tuning2_recall_results_{timestamp}.csv", index=False)

if len(threshold_results) > 0:
    pd.DataFrame(threshold_results).to_csv(f"{output_dir}/threshold_optimization_{timestamp}.csv", index=False)

if len(ensemble_results) > 0:
    pd.DataFrame(ensemble_results).to_csv(f"{output_dir}/ensemble_results_{timestamp}.csv", index=False)

print(f"\n💾 ARQUIVOS SALVOS:")
print(f"✅ tuning2_recall_results_{timestamp}.csv")
print(f"✅ threshold_optimization_{timestamp}.csv")
print(f"✅ ensemble_results_{timestamp}.csv")

print(f"\n🚀 PRÓXIMOS PASSOS:")
print(f"1. Escolher melhor modelo/ensemble para MÁXIMO RECALL")
print(f"2. Treinar modelo final no dataset completo")
print(f"3. VALIDAÇÃO FINAL no DatasetTest.csv")
print(f"4. Implementação clínica do 'detector paranóico'")

print(f"\n🏁 TUNING 2.0 RECALL-FOCUSED CONCLUÍDO!")
print(f"🎯 MISSÃO: Não perder nenhum bebê com problema nutricional!")

🚀 TUNING 2.0 - RECALL FOCUSED + ENSEMBLE PARANÓICO
🎯 OBJETIVO: 90%+ RECALL - NÃO PERDER NENHUM BEBÊ!
✅ Dados carregados: (3429, 39)
✅ Target distribution: {0: 2580, 1: 849}
✅ Classe minoritária: 24.8% (849 bebês)
✅ Features selecionadas: ['imc_pre_gravidico', 'k06_peso_engravidar', 'h02_peso']
📊 Estratégia: Máxima sensibilidade com apenas 3 features
✅ Dataset preprocessado: (3429, 3)
✅ Features escaladas para SVM

🧪 ALGORITMOS PARA TUNING RECALL:
  - SVC_linear: 20 combinações
  - Ridge_Recall: 20 combinações
  - LogisticRegression_Recall: 40 combinações
  - RandomForest_Recall: 162 combinações

🔥 INICIANDO TUNING RECALL-FOCUSED...
⏱️ Tempo estimado: 60-120 minutos

🧪 COMEÇANDO TUNING...

🔧 1/4 | Tuning SVC_linear para MÁXIMO RECALL
   ✅ RECALL: 1.0000 | F1: 0.3969 | AUC: 0.5191
   ⏱️  Tempo: 892.8s
   🎯 Params: {'C': 0.01, 'class_weight': {0: 1, 1: 5}}

🔧 2/4 | Tuning Ridge_Recall para MÁXIMO RECALL
   ✅ RECALL: 1.0000 | F1: 0.3974 | AUC: 0.6186
   ⏱️  Tempo: 0.2s
   🎯 Params: {'alpha

/opt/anaconda3/envs/pycaret_env/lib/python3.8/site-packages/sklearn/model_selection/_validation.py:794: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/opt/anaconda3/envs/pycaret_env/lib/python3.8/site-packages/sklearn/metrics/_scorer.py", line 76, in _cached_call
    return cache[method]
KeyError: 'predict'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/anaconda3/envs/pycaret_env/lib/python3.8/site-packages/sklearn/metrics/_scorer.py", line 115, in __call__
    score = scorer._score(cached_call, estimator, *args, **kwargs)
  File "/opt/anaconda3/envs/pycaret_env/lib/python3.8/site-packages/sklearn/metrics/_scorer.py", line 276, in _score
    y_pred = method_caller(estimator, "predict", X)
  File "/opt/anaconda3/envs/pycaret_env/lib/python3.8/site-packages/sklearn/metrics/_scorer.py", line 78, in _cache

   ✅ RECALL: nan
   📊 F1: nan | AUC: nan

🧪 Testando voting_hard
   ✅ RECALL: 1.0000
   📊 F1: 0.3973 | AUC: nan

🏆 RESUMO FINAL - MÁXIMO RECALL ACHIEVED
🥇 MELHOR MODELO INDIVIDUAL:
   SVC_linear
   RECALL: 1.0000 (±0.000)
   F1: 0.3969 | AUC: 0.5191

🎯 MELHOR COM THRESHOLD OTIMIZADO:
   SVC_linear com threshold 0.10
   RECALL: 1.0000
   F1: 0.3972

🤖 MELHOR ENSEMBLE:
   Estratégia: voting_soft
   RECALL: nan
   F1: nan

👶 IMPACTO MÉDICO:
   Dos 849 bebês com problema nutricional:
   ✅ DETECTADOS: 849 bebês (100.0%)
   ❌ PERDIDOS: 0 bebês (0.0%)
   🎯 META ATINGIDA: Excelente sensibilidade para triagem médica!

💾 ARQUIVOS SALVOS:
✅ tuning2_recall_results_20250721_162659.csv
✅ threshold_optimization_20250721_162659.csv
✅ ensemble_results_20250721_162659.csv

🚀 PRÓXIMOS PASSOS:
1. Escolher melhor modelo/ensemble para MÁXIMO RECALL
2. Treinar modelo final no dataset completo
3. VALIDAÇÃO FINAL no DatasetTest.csv
4. Implementação clínica do 'detector paranóico'

🏁 TUNING 2.0 RECALL-FOCUSED C

/opt/anaconda3/envs/pycaret_env/lib/python3.8/site-packages/sklearn/model_selection/_validation.py:794: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/opt/anaconda3/envs/pycaret_env/lib/python3.8/site-packages/sklearn/metrics/_scorer.py", line 76, in _cached_call
    return cache[method]
KeyError: 'decision_function'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/anaconda3/envs/pycaret_env/lib/python3.8/site-packages/sklearn/metrics/_scorer.py", line 373, in _score
    y_pred = method_caller(clf, "decision_function", X)
  File "/opt/anaconda3/envs/pycaret_env/lib/python3.8/site-packages/sklearn/metrics/_scorer.py", line 78, in _cached_call
    result = getattr(estimator, method)(*args, **kwargs)
AttributeError: 'VotingClassifier' object has no attribute 'decision_function'

During handling of the above